In [ ]:
%%writefile benchmark.c

#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
#include <time.h>

// Mesurer le temps d'exécution d'une boucle for
double measure_for_loop_time(int iterations) {
    double start_time = omp_get_wtime();
    for (int i = 0; i < iterations; ++i) {
    }
    double end_time = omp_get_wtime();
    return end_time - start_time;
}

// Mesurer le temps d'exécution d'une condition if
double measure_if_condition_time(int iterations) {
    double start_time = omp_get_wtime();
    for (int i = 0; i < iterations; ++i) {
        if (i % 2 == 0) {
        }
    }
    double end_time = omp_get_wtime();
    return end_time - start_time;
}

// Mesurer le temps d'exécution d'une directive OpenMP
double measure_omp_parallel_time(int iterations) {
    double start_time = omp_get_wtime();
    #pragma omp parallel for
    for (int i = 0; i < iterations; ++i) {
    }
    double end_time = omp_get_wtime();
    return end_time - start_time;
}

int main() {
    int iterations = 1000000;

    double for_time = measure_for_loop_time(iterations);
    double if_time = measure_if_condition_time(iterations);
    double omp_time = measure_omp_parallel_time(iterations);

    printf("Temps d'exécution pour une boucle for: %e secondes par itération\n", for_time / iterations);
    printf("Temps d'exécution pour une condition if: %e secondes par itération\n", if_time / iterations);
    printf("Temps d'exécution pour une directive OpenMP: %e secondes par itération\n", omp_time / iterations);

    return 0;
}


Overwriting benchmark.c


In [ ]:
!g++ -fopenmp -lopenblas benchmark.c -o benchmark

In [ ]:
!./benchmark

Temps d'exécution pour une boucle for: 2.730201e-09 secondes par itération
Temps d'exécution pour une condition if: 2.602876e-09 secondes par itération
Temps d'exécution pour une directive OpenMP: 1.532483e-08 secondes par itération


In [48]:
import re
import multiprocessing

def count_loops(filename):
    counts = {"for": 0, "if": 0, "omp": 0}

    with open(filename, "r") as f:
        for line in f:
            # Boucles for
            if re.search(r"for\s*\(", line):
                counts["for"] += 1

            # Boucles if
            if re.search(r"if\s*\(", line):
                counts["if"] += 1

            # Directives OpenMP
            if re.search(r"#pragma\s+omp\s+for", line):
                counts["omp"] += 1
            elif re.search(r"#pragma\s+omp\s+parallel", line):
                counts["omp"] += 1
            elif re.search(r"#pragma\s+omp", line):
                counts["omp"] += 1

    return counts

def extract_matrix_sizes(filename):
    matrix_sizes = []
    with open(filename, "r") as f:
        for line in f:
            match = re.search(r"int\s+matrix_sizes\[\]\s*=\s*\{(.+?)\};", line)
            if match:
                sizes = match.group(1).split(',')
                matrix_sizes = [int(size.strip()) for size in sizes]
    return matrix_sizes

def estimate_execution_time(counts, matrix_sizes):
    # Estimations de temps simplifiées pour chaque type de boucle
    base_time_per_for = 2.730201e-09
    base_time_per_if = 2.602876e-09
    base_time_per_omp = 1.532483e-08

    # Nombre de threads (le nombre de cœurs CPU)
    num_threads = multiprocessing.cpu_count()
    print(f"Nombre de threads: {num_threads}")


    # Calcul du temps total basé sur le nombre de boucles
    total_for_time = counts["for"] * base_time_per_for
    total_if_time = counts["if"] * base_time_per_if
    total_omp_time = counts["omp"] * base_time_per_omp

    # Réduction du temps en fonction de la parallélisation OpenMP
    if counts["omp"] > 0:
        total_omp_time /= num_threads  # Diviser par le nombre de threads
    matrix_size_factor = sum(matrix_sizes) / len(matrix_sizes) if matrix_sizes else 1

    total_time = (total_for_time + total_if_time + total_omp_time)*matrix_size_factor
    return total_time

if __name__ == "__main__":
    filename = "code3.c"
    counts = count_loops(filename)
    matrix_sizes = extract_matrix_sizes(filename)

    print(f"Nombre de boucles for: {counts['for']}")
    print(f"Nombre de conditions if: {counts['if']}")
    print(f"Nombre de directives OpenMP: {counts['omp']}")
    print(f"Taille des matrices: {matrix_sizes}")

    estimated_time = estimate_execution_time(counts, matrix_sizes)
    print(f"Temps d'exécution estimé: {estimated_time} secondes")


Nombre de boucles for: 4
Nombre de conditions if: 0
Nombre de directives OpenMP: 0
Taille des matrices: [3000]
Nombre de threads: 2
Temps d'exécution estimé: 3.2762411999999996e-05 secondes


In [74]:
def calculate_error(predicted_time, real_time):
    # Calcul de l'erreur relative
    error = abs(predicted_time - real_time) / real_time
    return error

def main():
    # Temps prédit et temps réel (en secondes)
    predicted_time =3.05782512e-05
    real_time =0.038632

    # Calcul de l'erreur
    error = calculate_error(predicted_time, real_time)

    # Affichage du résultat
    print(f"Temps prédit : {predicted_time} secondes")
    print(f"Temps réel : {real_time} secondes")
    print(f"Erreur relative : {error:.2f}")

if __name__ == "__main__":
    main()


Temps prédit : 3.05782512e-05 secondes
Temps réel : 0.038632 secondes
Erreur relative : 1.00


In [ ]:
%%writefile code.c

#include <iostream>
#include <fstream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include <omp.h>

void complex_computation(int matrix_size, std::vector<std::pair<int, double>>& results) {
    double result = 0.0;
    for (int i = 0; i < matrix_size * matrix_size; ++i) {
        result += static_cast<double>(rand()) / RAND_MAX;
    }

    double start_time = omp_get_wtime();

    #pragma omp sections
    {
        #pragma omp section
        {
            // First section of computation
            for (int i = 0; i < matrix_size * matrix_size / 2; ++i) {
                result = result * 2.0 + i;
            }
        }

        #pragma omp section
        {
            // Second section of computation
            for (int i = matrix_size * matrix_size / 2; i < matrix_size * matrix_size; ++i) {
                result = result * 2.0 + i;
            }
        }
    }

    double end_time = omp_get_wtime();
    double execution_time = end_time - start_time;

    #pragma omp critical
    results.push_back(std::make_pair(matrix_size, execution_time));
}

int main() {
    int matrix_sizes[] = {500, 600, 700, 800, 900, 1000, 1100,1200,1300,1400,1500,1600,1700,1800,1900,2000,2100,2200,2300,2400,2500,2600,2700,2800,2900,3000};

    int num_sizes = sizeof(matrix_sizes) / sizeof(matrix_sizes[0]);

    std::vector<std::pair<int, double>> results;

    #pragma omp parallel for
    for (int i = 0; i < num_sizes; ++i) {
        complex_computation(matrix_sizes[i], results);
    }

    std::ofstream csvfile("resultats_omp_sections.csv");
    csvfile << "Taille de la matrice, Temps d'exécution (s)\n";

    for (const auto& result : results) {
        csvfile << result.first << "," << result.second << "\n";
    }

    std::cout << "Résultats enregistrés dans le fichier 'resultats_omp_sections.csv'" << std::endl;

    return 0;
}




Writing code.c


In [ ]:
!g++ -fopenmp -lopenblas code.c -o code

In [ ]:
!./code

Résultats enregistrés dans le fichier 'resultats_omp_sections.csv'


In [ ]:
!cat resultats_omp_sections.csv

Taille de la matrice, Temps d'exécution (s)
1800,1.0227e-05
500,0.0635941
1900,1.189e-05
600,0.0723307
2000,1.212e-05
700,0.090892
2100,1.7084e-05
800,0.079625
2200,1.4327e-05
900,0.0757213
2300,1.0843e-05
1000,0.0938611
2400,8.599e-06
1100,0.102165
2500,1.4754e-05
1200,0.102057
1300,0.117725
2600,0.00195948
2700,9.532e-06
1400,0.105265
2800,1.155e-05
1500,0.122723
2900,8.746e-06
1600,0.129709
3000,7.985e-06
1700,0.149285


In [18]:
%%writefile code3.cpp

#include <iostream>
#include <fstream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include <chrono>

void complex_computation(int matrix_size, std::vector<std::pair<int, double>>& results) {
    double result = 0.0;
    for (int i = 0; i < matrix_size * matrix_size; ++i) {
        result += static_cast<double>(rand()) / RAND_MAX;
    }

    auto start_time = std::chrono::high_resolution_clock::now();
    for (int i = 0; i < matrix_size * matrix_size; ++i) {
        result = result * 2.0 + i;
    }
    auto end_time = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> execution_time = end_time - start_time;

    results.push_back(std::make_pair(matrix_size, execution_time.count()));
}

int main() {
    int matrix_sizes[] = {500, 600, 700, 800, 900, 1000, 1100,1200,1300,1400,1500,1600,1700,1800,1900,2000,2100,2200,2300,2400,2500,2600,2700,2800,2900,3000};

    int num_sizes = sizeof(matrix_sizes) / sizeof(matrix_sizes[0]);

    std::vector<std::pair<int, double>> results;

    for (int i = 0; i < num_sizes; ++i) {
        complex_computation(matrix_sizes[i], results);
    }

    std::ofstream csvfile("resultats_sequentiel.csv");
    csvfile << "Taille de la matrice,Temps d'exécution (s)\n";

    for (const auto& result : results) {
        csvfile << result.first << "," << result.second << "\n";
    }

    std::cout << "Résultats enregistrés dans le fichier 'resultats_sequentiel.csv'" << std::endl;

    return 0;
}


Overwriting code3.cpp


In [19]:
!g++ -o  code3 code3.cpp

In [20]:
!./code3

Résultats enregistrés dans le fichier 'resultats_sequentiel.csv'
